<a href="https://colab.research.google.com/github/victorm0202/temas_selectos_CD_2026/blob/main/Notebooks/Multimodal_IMDB_dataset_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![fig](https://storage.googleapis.com/kaggle-datasets-images/4189989/7235569/0db85036f2d850cc786e22259a76fbe1/dataset-cover.jpg?t=2023-12-19-06-31-49)

# MM-IMDb · Clasificación multimodal con ResNet-18 y BERT

En éste ejercicio, abordaremos un problema de clasificación con datos multimodales con las siguientes características:

- Modalidades: · Texto (sinopsis de películas) + imagen (póster)
- Categorías: 23 géneros que trataremos con un enfoque (*multi-etiqueta*), es decir, una película puede tener más de un género.

Vamos a comparar dos baselines unimodales y las tres estrategias de fusión que vimos en clase:

| Modelo | Integración |
|---|---|
| ResNet-18 | Predice géneros a partir del póster (baseline). |
| BERT | Predice géneros a partir de la sinopsis (baseline). |
| **Early fusion** | Concatena los vectores de características de ambos encoders y aplica un clasificador. |
| **Intermediate fusion** | Proyecta cada vector a un espacio de dimensión `FUSION_DIM`, promedia las proyecciones y clasifica. |
| **Late fusion** | Combina las probabilidades de dos clasificadores unimodales. |

La configuración inicial (para fines didácticos) ejecuta los modelos en un subconjunto de datos con los encoders congelados.

## Dependencias y configuración

Ejecútalo si tu environment no tiene estas librerias.

Ajusta `QUICK_MODE`, `N_QUICK` y `EPOCHS` según los recursos disponibles.

In [ ]:
# %pip install -U torch torchvision transformers datasets scikit-learn pillow pandas matplotlib tqdm
# Opcional (para división multilabel estratificada):
# %pip install iterative-stratification

In [ ]:
import random, copy, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
import platform

if platform.system() == 'Darwin': #si es macOS
    av1 = torch.backends.mps.is_available()
    # this ensures that the current PyTorch installation was built with MPS activated.
    av2 = torch.backends.mps.is_built()
    if(av1==av2==True):
        print('MPS available?',av1,'MPS built?',av2)
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
else:
    #Check if GPU is available
    if torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')

In [ ]:
DEVICE = device
print('Device:', DEVICE)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

REPO_ID = 'pranavmr/MM-IMDb'
QUICK_MODE = True      # False = conjunto completo (más tiempo/espacio)
N_QUICK = 1200         # Para pruebas iniciales; suba a 3000 si hay clases escasas
BATCH_SIZE = 8
EPOCHS = 2            # Aumentar para entrenamientos más largos
LR = 2e-3             # Tasa de aprendizaje de las cabezas, encoders congelados
MAX_LENGTH = 128
NUM_WORKERS = 0       # Compatible con notebooks/macOS
FREEZE_ENCODERS = True  # False permite actualizar también ResNet y BERT

FUSION_DIM = 256      # Dimensión compartida para intermediate fusion

## Cargar los datos

La versión consultada de Hugging Face contiene los campos `image`, `text` y `labels`. Esta celda comprueba el esquema en ejecución. Los datos se descargan desde el repositorio remoto; si el repositorio cambia, habrá que ajustar la lectura.

In [ ]:
if QUICK_MODE:
    stream = load_dataset(REPO_ID, split='train', streaming=True)
    records = list(stream.shuffle(seed=SEED, buffer_size=3000).take(N_QUICK))
else:
    full = load_dataset(REPO_ID, split='train')
    records = [full[i] for i in range(len(full))]

print('Registros leídos:', len(records))
print('Columnas:', list(records[0]))
required = {'image', 'text', 'labels'}
assert required.issubset(records[0]), f'Faltan campos: {required-set(records[0])}'

def normalize_labels(x):
    if isinstance(x, (list, tuple)) and all(isinstance(v, str) for v in x):
        return list(x)
    raise ValueError(f'Formato de etiquetas no esperado: {type(x)} / {str(x)[:100]}')

records = [r for r in records if isinstance(r['text'], str) and r['text'].strip() and r['image'] is not None and normalize_labels(r['labels'])]
labels_list = [normalize_labels(r['labels']) for r in records]
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels_list).astype('float32')
print('Registros válidos:', len(records), '| Etiquetas observadas:', len(mlb.classes_))
print('Géneros:', list(mlb.classes_))
assert len(records) > 20


> ### 📝 **Ejercicio Práctico**
>
> **Instrucciones:** Realiza un análisis exploratorio del dataset en ambas modalidades para ver cosas como: ejemplos de la modalidad imágen y texto, distribución de las categorías, cantidad de géneros por película, longitud de las sinopsis, entre otras que consideres relevantes.
>
> *⏱️ Tiempo estimado: 10 minutos*

## Particiones de los datos

Se realiza **estratificación multietiqueta** si está instalada `iterative-stratification`; en otro caso usamos una partición aleatoria con semilla fija.

La estratificación multilabel ayuda a preservar frecuencias marginales, pero no garantiza representación de géneros extremadamente raros en cada split.

In [ ]:
indices = np.arange(len(records))
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    def split_ids(idx, test_size, seed):
        splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        a, b = next(splitter.split(idx.reshape(-1,1), Y[idx]))
        return idx[a], idx[b]
    print('Partición multilabel estratificada.')
except ImportError:
    warnings.warn('iterative-stratification no instalado: división aleatoria simple.')
    def split_ids(idx, test_size, seed):
        return train_test_split(idx, test_size=test_size, random_state=seed, shuffle=True)

idx_train, idx_other = split_ids(indices, .30, SEED)
idx_val, idx_test = split_ids(np.asarray(idx_other), .50, SEED+1)
for name, idx in [('train',idx_train),('val',idx_val),('test',idx_test)]:
    print(name, len(idx), 'positivos por clase (mín/máx):', int(Y[idx].sum(0).min()), int(Y[idx].sum(0).max()))
assert not (set(idx_train)&set(idx_val) or set(idx_train)&set(idx_test) or set(idx_val)&set(idx_test))

## Dataset de PyTorch y transformaciones

Se utiliza el preprocesamiento estándar de ResNet (224 × 224, normalización ImageNet) y el tokenizador de BERT. `MAX_LENGTH=128` trunca las sinopsis largas: es una decisión práctica que puede estudiarse experimentalmente.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
image_tf = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize(mean=[.485,.456,.406], std=[.229,.224,.225])
])

class MovieDataset(Dataset):
    def __init__(self, indices): self.indices = list(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]; r = records[j]
        im = r['image']
        if not isinstance(im, Image.Image):
            if isinstance(im, dict) and im.get('bytes'):
                import io
                im = Image.open(io.BytesIO(im['bytes']))
            else: im = Image.open(im['path'] if isinstance(im, dict) else im)
        tok = tokenizer(r['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
        return {'pixel_values': image_tf(im.convert('RGB')),
                'input_ids':tok['input_ids'].squeeze(0),
                'attention_mask':tok['attention_mask'].squeeze(0),
                'labels':torch.tensor(Y[j])}

train_loader = DataLoader(MovieDataset(idx_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(MovieDataset(idx_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(MovieDataset(idx_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print({k:v.shape for k,v in next(iter(train_loader)).items()})

## Encoders y modelos de clasificación

ResNet-18 genera **512 características** por imagen y BERT-base-uncased genera **768 características** por sinopsis (vector del primer token, `[CLS]`). Por defecto se mantienen congelados: el entrenamiento actualiza las capas de clasificación y fusión.

### Estrategias de fusión

- **Early fusion:** concatena ambos embeddings (`512 + 768`) y los pasa a una red clasificadora.
- **Intermediate fusion:** aprende dos proyecciones lineales, una de `512 → d` y otra de `768 → d`, con `d = FUSION_DIM`. Combina los resultados mediante un promedio y aplica una capa de clasificación:

  $$z_v=W_v h_v+b_v,\qquad z_t=W_t h_t+b_t,\qquad z=\frac{z_v+z_t}{2},\qquad \mathrm{logits}=W_c z+b_c.$$

  Las proyecciones se entrenan conjuntamente para la tarea.
- **Late fusion:** entrena una cabeza visual y otra textual; sus probabilidades se promedian con un peso ajustado en validación.

`FUSION_DIM` permite cambiar la dimensión del espacio de fusión sin modificar el resto del código.

In [ ]:
class Encoders(nn.Module):
    def __init__(self, freeze=True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT
        self.cnn = models.resnet18(weights=weights)
        self.cnn.fc = nn.Identity()  # 512
        self.bert = AutoModel.from_pretrained('bert-base-uncased')
        self.freeze = freeze
        if freeze:
            for p in self.cnn.parameters(): p.requires_grad = False
            for p in self.bert.parameters(): p.requires_grad = False
    def forward(self, batch):
        # Sin gradientes ni dropout en los encoders congelados.
        if self.freeze:
            self.cnn.eval(); self.bert.eval()
        with torch.set_grad_enabled(not self.freeze and torch.is_grad_enabled()):
            image_features = self.cnn(batch['pixel_values'])
            text_features = self.bert(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).last_hidden_state[:,0,:]
        return image_features, text_features

class MovieClassifier(nn.Module):
    def __init__(self, mode, n_labels, freeze=True):
        super().__init__()
        assert mode in {'image','text','early','intermediate','late'}
        self.mode = mode
        self.enc = Encoders(freeze)
        if mode == 'image': self.head_img = nn.Linear(512,n_labels)
        if mode == 'text': self.head_txt = nn.Linear(768,n_labels)
        if mode == 'early':
            self.head = nn.Sequential(nn.Linear(512+768,256),nn.ReLU(),nn.Dropout(.2),nn.Linear(256,n_labels))
        if mode == 'intermediate':
            self.img_proj = nn.Linear(512,FUSION_DIM)
            self.txt_proj = nn.Linear(768,FUSION_DIM)
            self.head = nn.Linear(FUSION_DIM,n_labels)
        if mode == 'late':
            self.head_img = nn.Linear(512,n_labels)
            self.head_txt = nn.Linear(768,n_labels)
    def forward(self,batch):
        im,txt = self.enc(batch)
        if self.mode == 'image': return self.head_img(im)
        if self.mode == 'text': return self.head_txt(txt)
        if self.mode == 'early': return self.head(torch.cat([im,txt],dim=1))
        if self.mode == 'intermediate':
            visual_z = self.img_proj(im)
            text_z = self.txt_proj(txt)
            fused = (visual_z + text_z) / 2
            return self.head(fused)
        return self.head_img(im), self.head_txt(txt)  # late: dos logits


## 5. Entrenamiento y evaluación

Se usa `BCEWithLogitsLoss` (adecuada para multilabel). Para *late fusion*, la pérdida es el promedio de las pérdidas de las dos cabezas; durante la inferencia se combinan sus **probabilidades**. El mejor estado se guarda según la pérdida de validación. Para simplificar, las predicciones usan umbral 0.5; en un estudio formal se puede calibrar ese umbral usando **solo validación**.

> ### 📝 **Ejercicio Práctico**
>
> **Instrucciones:** Averigua qué es y cómo se define la función de costo BCEWithLogitsLoss. ¿En qué se diferencia de la Cross Entropy Loss que has usado tradicionalmente?
>
> *⏱️ Tiempo estimado: 5 minutos*

In [ ]:
def move(batch): return {k:v.to(DEVICE) for k,v in batch.items()}

def loss_for(output, y, criterion):
    if isinstance(output, tuple): return .5*(criterion(output[0],y)+criterion(output[1],y))
    return criterion(output,y)

def fit_model(mode):
    print('\nEntrenando:', mode)
    model = MovieClassifier(mode, Y.shape[1], freeze=FREEZE_ENCODERS).to(DEVICE)
    opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=LR)
    criterion = nn.BCEWithLogitsLoss()
    best_loss = float('inf'); best_state = None
    for epoch in range(EPOCHS):
        model.train(); total = 0
        for raw in tqdm(train_loader,desc=f'{mode} epoch {epoch+1}',leave=False):
            batch = move(raw); opt.zero_grad()
            out = model(batch); loss = loss_for(out,batch['labels'],criterion)
            loss.backward(); opt.step(); total += loss.item()*len(batch['labels'])
        model.eval(); vtotal = 0
        with torch.no_grad():
            for raw in val_loader:
                batch=move(raw); out=model(batch)
                vtotal += loss_for(out,batch['labels'],criterion).item()*len(batch['labels'])
        vloss = vtotal/len(idx_val)
        print(f'Epoch {epoch+1}: train={total/len(idx_train):.4f} val={vloss:.4f}')
        if vloss < best_loss:
            best_loss=vloss
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(model, loader):
    model.eval(); ys=[]; outs=[]
    for raw in loader:
        batch=move(raw); out=model(batch)
        ys.append(batch['labels'].cpu().numpy())
        if isinstance(out,tuple):
            outs.append(tuple(torch.sigmoid(o).cpu().numpy() for o in out))
        else: outs.append(torch.sigmoid(out).cpu().numpy())
    y=np.concatenate(ys)
    if model.mode=='late': return y, tuple(np.concatenate([v[i] for v in outs]) for i in (0,1))
    return y,np.concatenate(outs)

def scores(y, prob, threshold=.5):
    pred=(prob>=threshold).astype(int)
    return {'micro_f1':f1_score(y,pred,average='micro',zero_division=0),
            'macro_f1':f1_score(y,pred,average='macro',zero_division=0)}


## 6. Ejecutar los cinco modelos

Esta celda puede tardar bastante, incluso con `QUICK_MODE=True`, porque BERT y ResNet se ejecutan para cada lote. En una GPU pequeña pruebe `BATCH_SIZE=4`; en CPU reduzca `N_QUICK` y `EPOCHS`. Si el dataset se vuelve a cargar, reinicie desde el principio para conservar la correspondencia entre índices y etiquetas.

**Consejo:** para una primera prueba rápida, ejecute solo `['image','text','early']` y luego habilite los otros dos.

In [ ]:
MODES = ['image','text','early','intermediate','late']
models_fit = {}
for mode in MODES:
    model = fit_model(mode)
    models_fit[mode] = model
    # En CPU los modelos también consumen RAM; en GPU liberar caché entre entrenamientos.
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()


## 7. Selección del peso de late fusion en validación

El peso `alpha` multiplica la probabilidad visual, y `1-alpha` la textual. Se elige usando **Macro-F1 de validación**, sin consultar la prueba.

In [ ]:
alpha = .5
if 'late' in models_fit:
    yv, (pv_img,pv_txt) = predict(models_fit['late'], val_loader)
    candidates=np.linspace(0,1,11)
    alpha=max(candidates,key=lambda a:scores(yv,a*pv_img+(1-a)*pv_txt)['macro_f1'])
    print('Alpha seleccionado en validación:',alpha)


## 8. Resultados en prueba

El conjunto de prueba se utiliza **solo después** de elegir hiperparámetros y el peso de late fusion. La tabla compara métricas sobre exactamente los mismos ejemplos. Las puntuaciones pueden ser bajas con pocas épocas, muestras reducidas y encoders congelados.

In [ ]:
results=[]; predictions={}
for mode, model in models_fit.items():
    yt, probs = predict(model,test_loader)
    if mode=='late': probs=alpha*probs[0]+(1-alpha)*probs[1]
    predictions[mode]=probs
    results.append({'modelo':mode,**scores(yt,probs)})
results_df=pd.DataFrame(results).sort_values('macro_f1',ascending=False)
display(results_df)
if not results_df.empty:
    results_df.set_index('modelo')[['micro_f1','macro_f1']].plot.bar(figsize=(8,4),ylim=(0,1),rot=0)
    plt.title('MM-IMDb: evaluación en test (partición docente)'); plt.tight_layout(); plt.show()


## 9. Métricas por género y análisis de errores

Se presenta el reporte por etiqueta del modelo con mejor Macro-F1. Los géneros poco frecuentes pueden obtener F1=0; esto no significa necesariamente que el modelo sea defectuoso: revise cuántos ejemplos positivos hay por categoría.

In [ ]:
best_mode=results_df.iloc[0]['modelo']
print('Mejor modelo (macro-F1 en test, SOLO descriptivo):',best_mode)
print(classification_report(yt,(predictions[best_mode]>=.5).astype(int),target_names=mlb.classes_,zero_division=0))


## 10. Guardar resultados y reproducibilidad

Se guardan métricas, nombres de etiquetas, índices del subconjunto y configuración. **Advertencia:** los índices son posiciones dentro de los datos leídos y no equivalen a IDs oficiales de MM-IMDb. Para reusar exactamente los splits, preserve el mismo origen, orden de lectura y semilla; para un experimento publicable recupere los splits oficiales por identificador de película.


In [ ]:
import json
from pathlib import Path
OUT=Path('resultados_mmimdb'); OUT.mkdir(exist_ok=True)
results_df.to_csv(OUT/'metricas_test.csv',index=False)
with open(OUT/'configuracion.json','w',encoding='utf-8') as f:
    json.dump({'repo_id':REPO_ID,'seed':SEED,'quick_mode':QUICK_MODE,'n_records':len(records),
               'freeze_encoders':FREEZE_ENCODERS,'epochs':EPOCHS,'batch_size':BATCH_SIZE,
               'max_length':MAX_LENGTH,'alpha_late':float(alpha),
               'classes':mlb.classes_.tolist(),
               'indices':{'train':list(map(int,idx_train)),'val':list(map(int,idx_val)),'test':list(map(int,idx_test))}},f,indent=2)
print('Archivos:',list(OUT.iterdir()))

> ### 📝 **Ejercicio Práctico**
>
> **Instrucciones:** Otra forma de realizar la fusión intermedia es mediante ´compuertas de fusión´ (gated fusion). Investiga en qué consiste ésta estrategia, sus formulación matemática e interpretación. Básate en el artículo

> - Arevalo et al. (2017), *Gated Multimodal Units for Information Fusion*: https://arxiv.org/abs/1702.01992
>
> *⏱️ Tiempo estimado: 10 minutos*

> ### 📝 **Ejercicio Práctico**
>
> **Instrucciones:** Implementa variantes de las arquitecturas usadas:
>
> - Implementando gated fusion
> - Otras variantes de la arquitectura o parámetros (tamaño del dataset, épocas, etc...)
>
> Compara los resultados con éste procedimiento estándar.

> *⏱️ Tiempo estimado: 15 minutos*